<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-05-intro-to-rag/notebooks/Ophelie_Chunking_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chunking & Embeddings in LlamaIndex

**Task: Implementing Chunking & Embeddings in LlamaIndex (Mandatory)**

This notebook explores how chunking strategy affects retrieval quality in a RAG pipeline. We'll load a sample document, try three chunking strategies (fixed-length, overlapping, semantic), generate embeddings, and use Gemini to answer questions from the retrieved chunks.

## Step 1: Install required libraries

Note: the task instructions specify `llama-index-llms-gemini`, but in the previous chatbot task in this externship that package hit a 429 quota-exceeded error (it's built on the now-deprecated `google.generativeai` SDK). This notebook uses `llama-index-llms-google-genai` instead, which is the current recommended integration.

In [ ]:
!pip install -q llama-index llama-index-embeddings-huggingface llama-index-llms-google-genai chromadb llama-index-vector-stores-chroma scikit-learn nest_asyncio google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 707.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69

## Step 2: Load a Sample Document (Plain Text)

Instead of uploading a file, we generate a sample document directly in the notebook so it's fully self-contained and reproducible. It's a fictional pharmaceutical product document with several distinct sections (overview, manufacturing, storage, safety, regulatory) so we can actually see topic shifts when we get to semantic chunking.

In [ ]:
import os

os.makedirs("sample_docs", exist_ok=True)

sample_text = """Product Overview

Cardavex-XR (extended-release fictarolol tablets) is a fictional cardiovascular
medication used in this exercise to demonstrate document chunking and retrieval.
It is formulated as a 24-hour extended-release tablet intended for once-daily
dosing. Each tablet contains 50mg, 100mg, or 200mg of the active ingredient,
fictarolol hydrochloride, along with inactive excipients including
microcrystalline cellulose, hypromellose, and magnesium stearate. The tablet
coating is designed to control the release rate of the active ingredient over
a 24-hour period, maintaining stable plasma concentrations and reducing the
peak-to-trough fluctuation seen with immediate-release formulations.

Manufacturing Process

Cardavex-XR tablets are manufactured using a wet granulation process. The
active pharmaceutical ingredient is first blended with microcrystalline
cellulose in a high-shear granulator, then bound using an aqueous hypromellose
solution. The resulting granules are dried in a fluid-bed dryer until moisture
content falls below 2%, then milled to achieve a uniform particle size
distribution. Magnesium stearate is added as a lubricant before compression.
Tablets are compressed on a rotary tablet press and then coated with a
semi-permeable membrane that controls the osmotic release of the drug. Each
production batch undergoes in-process controls for tablet hardness, friability,
and dissolution rate before release to packaging.

Storage Conditions

Cardavex-XR tablets must be stored at controlled room temperature, between
20 degrees Celsius and 25 degrees Celsius, with excursions permitted between
15 degrees Celsius and 30 degrees Celsius. Tablets should be protected from
moisture and stored in the original container with the desiccant provided.
Do not store in bathrooms or other high-humidity environments. Once opened,
the container should be used within 90 days. Tablets should never be frozen,
as freezing can damage the extended-release coating and cause the medication
to release too quickly, which could lead to an overdose.

Safety Data

The most common adverse reactions reported in clinical trials of Cardavex-XR
were dizziness, fatigue, and mild headache, each occurring in fewer than 8% of
patients. Serious but rare adverse events included symptomatic hypotension and
bradycardia, primarily in patients over 65 or those taking concurrent
beta-blockers. Cardavex-XR is contraindicated in patients with severe hepatic
impairment, as fictarolol is primarily metabolized by the liver. Patients
should be advised not to crush, chew, or split the tablets, since this would
destroy the extended-release mechanism and could result in rapid absorption of
the full dose at once.

Regulatory Compliance

This fictional product document is formatted in the style of a real
pharmaceutical technical data sheet for the purposes of this exercise,
referencing the kind of information that would appear in a submission
consistent with 21 CFR Part 11 recordkeeping requirements and ICH Q1A
stability testing guidelines. In a real regulatory submission, this section
would also reference batch release certificates, stability study data across
multiple climate zones, and any applicable BSE/TSE statements for excipients
of animal origin.

Clinical Usage Notes

Cardavex-XR should be taken at approximately the same time each day, with or
without food. If a dose is missed, patients should take it as soon as they
remember unless it is within 12 hours of the next scheduled dose, in which
case the missed dose should be skipped. Abrupt discontinuation is not
recommended; the dose should be tapered over 1-2 weeks under physician
supervision to avoid rebound hypertension.
"""

with open("sample_docs/cardavex_xr.txt", "w") as f:
    f.write(sample_text)

from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("sample_docs").load_data()
print(f"Loaded {len(documents)} documents.")
print(f"Document length: {len(documents[0].text)} characters")

Loaded 1 documents.
Document length: 3700 characters


**Observations (no chunking):** the entire ~4,600-character document is loaded as a single `Document` object. If we handed this whole thing to the AI for every query, it would have to read the full text (storage conditions, safety data, manufacturing, etc.) even for a question about just one section. That's the problem chunking solves.

## Step 3: Apply Different Chunking Strategies

### Fixed-Length Chunking

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter_fixed = SentenceSplitter(chunk_size=300, chunk_overlap=0)  # No overlap
chunks_fixed = splitter_fixed.get_nodes_from_documents(documents)
print(f"Total Fixed-Length Chunks Created: {len(chunks_fixed)}")
print("\n--- First chunk preview ---")
print(chunks_fixed[0].text[:300])

Total Fixed-Length Chunks Created: 3

--- First chunk preview ---
Product Overview

Cardavex-XR (extended-release fictarolol tablets) is a fictional cardiovascular
medication used in this exercise to demonstrate document chunking and retrieval.
It is formulated as a 24-hour extended-release tablet intended for once-daily
dosing. Each tablet contains 50mg, 100mg, o


### Overlapping Chunking

In [ ]:
splitter_overlap = SentenceSplitter(chunk_size=300, chunk_overlap=50)  # 50-token overlap
chunks_overlap = splitter_overlap.get_nodes_from_documents(documents)
print(f"Total Overlapping Chunks Created: {len(chunks_overlap)}")
print("\n--- First chunk preview ---")
print(chunks_overlap[0].text[:300])

Total Overlapping Chunks Created: 4

--- First chunk preview ---
Product Overview

Cardavex-XR (extended-release fictarolol tablets) is a fictional cardiovascular
medication used in this exercise to demonstrate document chunking and retrieval.
It is formulated as a 24-hour extended-release tablet intended for once-daily
dosing. Each tablet contains 50mg, 100mg, o


### Semantic Chunking (Advanced)

In [ ]:
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Semantic chunking needs an embedding model to detect topic shifts
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
semantic_splitter = SemanticSplitterNodeParser(embed_model=embed_model)
chunks_semantic = semantic_splitter.get_nodes_from_documents(documents)
print(f"Total Semantic Chunks Created: {len(chunks_semantic)}")
print("\n--- First chunk preview ---")
print(chunks_semantic[0].text[:300])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total Semantic Chunks Created: 3

--- First chunk preview ---
Product Overview

Cardavex-XR (extended-release fictarolol tablets) is a fictional cardiovascular
medication used in this exercise to demonstrate document chunking and retrieval.
It is formulated as a 24-hour extended-release tablet intended for once-daily
dosing. Each tablet contains 50mg, 100mg, o


### Quick comparison across strategies

In [ ]:
print(f"{'Strategy':<20}{'# Chunks':<10}")
print(f"{'Fixed-length':<20}{len(chunks_fixed):<10}")
print(f"{'Overlapping':<20}{len(chunks_overlap):<10}")
print(f"{'Semantic':<20}{len(chunks_semantic):<10}")

Strategy            # Chunks  
Fixed-length        3         
Overlapping         4         
Semantic            3         


## Step 4: Generate Embeddings for Retrieval

We already created `embed_model` above for semantic chunking, so we reuse it here rather than loading it twice.

In [ ]:
for chunk in chunks_overlap:  # Using Overlapping Chunks for best retrieval
    chunk.embedding = embed_model.get_text_embedding(chunk.text)

print("Embeddings Generated Successfully!")
print(f"Embedding vector length: {len(chunks_overlap[0].embedding)}")

Embeddings Generated Successfully!
Embedding vector length: 384


## Step 5: Set up API key and Store & Retrieve Embeddings Using Gemini

As in the previous chatbot task, the Gemini API key is read from **Colab Secrets** (name `GOOGLE_API_KEY`) rather than hardcoded, so the key value never has to live inside the notebook file.

In [ ]:
import os
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("API key loaded:", "Yes" if GOOGLE_API_KEY else "No")

API key loaded: Yes


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from llama_index.core import VectorStoreIndex
from llama_index.llms.google_genai import GoogleGenAI

# Set up Gemini as our LLM for generating answers
# Using the rolling "-latest" alias: "models/gemini-2.0-flash" (from the task instructions)
# returned a 429 quota error and "models/gemini-2.5-flash" returned a 404 in the previous
# task in this externship, so we go straight to the alias that worked there.
llm = GoogleGenAI(model="models/gemini-flash-latest")

# Create an index with our embeddings (built from the overlapping chunks)
index = VectorStoreIndex(chunks_overlap, embed_model=embed_model)

# Set up query engine with Gemini
query_engine = index.as_query_engine(llm=llm, similarity_top_k=2)

# Test a retrieval query
response = query_engine.query("What is the document about?")
print(response)

Cardavex-XR is a fictional cardiovascular medication formulated as a once-daily, extended-release fictarolol tablet. The provided technical information details its product overview (dosage strengths and excipients), manufacturing process using wet granulation, safety data (including common and serious adverse reactions, contraindications, and handling warnings), regulatory compliance standards, and clinical usage guidelines.


**Observations:** with chunking & embeddings, the query engine retrieves only the most relevant chunk(s) (`similarity_top_k=2`) instead of the whole ~4,600-character document, and Gemini answers using just that retrieved context.

### A more targeted retrieval test

To actually see chunking earning its keep, let's ask a question that only one section of the document answers, and check which chunk got retrieved.

In [ ]:
response2 = query_engine.query("What temperature range should this product be stored at?")
print("Answer:", response2)
print("\n--- Retrieved chunk(s) ---")
for node in response2.source_nodes:
    print(f"[score={node.score:.3f}] {node.text[:200]}...\n")

Answer: Cardavex-XR tablets must be stored at controlled room temperature, between 20 degrees Celsius and 25 degrees Celsius, with excursions permitted between 15 degrees Celsius and 30 degrees Celsius.

--- Retrieved chunk(s) ---
[score=0.406] The resulting granules are dried in a fluid-bed dryer until moisture
content falls below 2%, then milled to achieve a uniform particle size
distribution. Magnesium stearate is added as a lubricant bef...

[score=0.210] Safety Data

The most common adverse reactions reported in clinical trials of Cardavex-XR
were dizziness, fatigue, and mild headache, each occurring in fewer than 8% of
patients. Serious but rare adve...



## Step 6: Notes

- Followed the task's chunking steps (fixed-length, overlapping, semantic) and printed chunk counts for each to compare them directly.
- The task instructions specify `llama-index-llms-gemini` and `models/gemini-2.0-flash`. Based on real errors encountered in the previous chatbot task in this externship (429 quota exceeded on the deprecated package, and a 404 on `gemini-2.5-flash`), this notebook uses `llama-index-llms-google-genai` with the `models/gemini-flash-latest` alias instead, plus `nest_asyncio` to avoid an event-loop conflict in Colab.
- Added a second, more targeted query ("What temperature range...") and printed the retrieved source chunk with its similarity score, to actually verify retrieval is pulling the relevant section rather than the whole document.
- Next task in this step: query processing & retrieval optimization.